# Ngôn Sắc Việt — Train Whisper (PhoWhisper-small) Audio H'Mông → Việt

Notebook này huấn luyện thử nghiệm mô hình **speech_translation** (`vinai/PhoWhisper-small`, nghe tiếng H'Mông → xuất câu tiếng Việt) từ bộ dữ liệu âm thanh công khai trên GPU Kaggle.

> Lưu ý: Theo quy tắc dự án, `import_speech.py` chỉ chạy `--allow-unknown-license` (KHÔNG tự đánh dấu `--mark-verified`), nên checkpoint sinh ra mang cờ `trained_on_unverified: true` để đo điểm WER/CER benchmark trước khi người bản ngữ nghe rà soát.

In [ ]:
REPO_URL = "https://github.com/minhduy6868/vietlangai-model.git"
BRANCH = "main"
BASE_MODEL = "vinai/PhoWhisper-small"
EPOCHS = 6
LR = 1e-5
BATCH = 16
GRAD_ACCUM = 4


In [ ]:
import shutil
import subprocess
import sys

import torch

smi = shutil.which("nvidia-smi")
if smi:
    print(subprocess.check_output([smi], text=True))
if not torch.cuda.is_available():
    raise RuntimeError("Không phát hiện GPU CUDA trên Kaggle!")
print("CUDA GPU:", torch.cuda.get_device_name(0))

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "av>=12", "transformers>=4.56,<5", "accelerate>=1.0", "pyarrow>=15", "sentencepiece>=0.2",
])


In [ ]:
import os
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "vietlangai-model"

if (REPO_DIR / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
print("Working dir:", Path.cwd())
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-u", "training/import_speech.py", "--allow-unknown-license"])


In [ ]:
import os
import subprocess
import sys

OUT_DIR = "/kaggle/working/whisper-hmn"
cmd = [
    sys.executable, "-u", "training/finetune_whisper.py",
    "--base", BASE_MODEL,
    "--task", "speech_translation",
    "--allow-unverified",
    "--epochs", str(EPOCHS),
    "--lr", str(LR),
    "--batch", str(BATCH),
    "--grad-accum", str(GRAD_ACCUM),
    "--output", OUT_DIR,
]
env = {**os.environ, "PYTHONUNBUFFERED": "1"}
print("Chạy:", " ".join(cmd), flush=True)
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env) as proc:
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)


In [ ]:
import shutil
from pathlib import Path

bundle_dir = Path("/kaggle/working/whisper-hmn")
ckpt_dir = bundle_dir / "checkpoints"
if (bundle_dir / "ngonsacviet.json").is_file() and ckpt_dir.exists():
    shutil.rmtree(ckpt_dir)

zip_file = shutil.make_archive("/kaggle/working/whisper-hmn-ready", "zip", root_dir="/kaggle/working", base_dir="whisper-hmn")
print("Đã đóng gói:", zip_file, Path(zip_file).stat().st_size)
